# ⚖️ Day 02a: CAP Theorem — Consistency, Availability, Partition Tolerance

---

## 🎯 What You'll Master Today
- CAP theorem: why you can only pick 2 out of 3
- CP vs AP vs CA trade-offs
- PACELC extension
- Real-world database classification

---

### 🎭 The Analogy

Imagine 3 bank branches (nodes). A customer deposits $100 at Branch A.  
Now a **network cable breaks** between branches (partition!).

- **CP**: Freeze all branches until cable is fixed. Data is consistent but unavailable.  
- **AP**: Keep all branches open, but they may show different balances temporarily.  
- **CA**: Only works if the cable NEVER breaks (no partition = single server).

```
╔══════════════════════════════════════════════════════════════════════╗
║                     CAP THEOREM TRIANGLE                            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║                        Consistency (C)                               ║
║                           /     \                                    ║
║                          /       \                                   ║
║                         /   CA    \                                  ║
║                        /  (single  \                                 ║
║                       /   server)   \                                ║
║                      / CP           \                                ║
║                     /                 \                              ║
║          Partition (P) ──── AP ──── Availability (A)                ║
║                                                                      ║
║   CP: MongoDB, HBase, ZooKeeper, Redis Cluster                      ║
║   AP: Cassandra, DynamoDB, CouchDB, Riak                           ║
║   CA: Single-node PostgreSQL, MySQL (no partition = no choice!)     ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. CAP Theorem Simulator
# ============================================

import random, time

class DistributedNode:
    def __init__(self, name):
        self.name = name
        self.data = {}       # key → value
        self.online = True
    
    def write(self, key, value):
        self.data[key] = value
    
    def read(self, key):
        return self.data.get(key, None)

class CAPSimulator:
    def __init__(self, mode="CP"):
        self.mode = mode
        self.nodes = [DistributedNode(f"Node-{i}") for i in range(3)]
        self.partition = False
    
    def create_partition(self):
        """Simulate network partition: Node-2 isolated."""
        self.partition = True
        print("  ⚡ NETWORK PARTITION: Node-2 is isolated!")
    
    def write(self, key, value):
        print(f"\n  WRITE({key}={value}):")
        
        if self.mode == "CP":
            # CP: Require ALL nodes to ack → reject if partition
            if self.partition:
                print("  ❌ WRITE REJECTED: Cannot reach all nodes (consistency first)")
                return False
            for node in self.nodes:
                node.write(key, value)
                print(f"    ✅ {node.name}: written")
            return True
        
        elif self.mode == "AP":
            # AP: Write to reachable nodes → accept even during partition
            for node in self.nodes:
                if node.name == "Node-2" and self.partition:
                    print(f"    ⚠️  {node.name}: unreachable (stale data!)")
                else:
                    node.write(key, value)
                    print(f"    ✅ {node.name}: written")
            return True
    
    def read_all(self, key):
        print(f"\n  READ({key}) from all nodes:")
        for node in self.nodes:
            val = node.read(key)
            if node.name == "Node-2" and self.partition:
                print(f"    {node.name}: {val} ⚠️  (isolated — may be stale)")
            else:
                print(f"    {node.name}: {val}")

# Demonstrate CP
print("═" * 60)
print("  CP MODE (Consistent + Partition-tolerant)")
print("═" * 60)
cp = CAPSimulator("CP")
cp.write("balance", 1000)
cp.create_partition()
cp.write("balance", 900)  # Will be REJECTED
cp.read_all("balance")

In [ ]:
# Demonstrate AP
print("═" * 60)
print("  AP MODE (Available + Partition-tolerant)")
print("═" * 60)
ap = CAPSimulator("AP")
ap.write("balance", 1000)
ap.create_partition()
ap.write("balance", 900)  # Writes to reachable nodes only
ap.read_all("balance")
print("\n  ⚠️  Node-2 still shows 1000 (stale!) — INCONSISTENT but AVAILABLE")

In [ ]:
# ============================================
# 2. Consistency Levels (Quorum)
# ============================================

print("  Consistency Levels — Tunable Consistency:\n")

class QuorumDB:
    def __init__(self, n_replicas=3):
        self.n = n_replicas
    
    def check_consistency(self, w, r):
        """W + R > N guarantees strong consistency."""
        strong = w + r > self.n
        return {
            "N": self.n, "W": w, "R": r,
            "W+R": w + r,
            "strong_consistency": strong,
            "write_availability": f"{self.n - w} nodes can fail",
            "read_availability": f"{self.n - r} nodes can fail",
        }

qdb = QuorumDB(n_replicas=3)

configs = [
    (1, 1, "ONE/ONE — fast, inconsistent"),
    (2, 2, "QUORUM/QUORUM — balanced ✅"),
    (3, 1, "ALL/ONE — slow writes, fast reads"),
    (1, 3, "ONE/ALL — fast writes, slow reads"),
]

for w, r, label in configs:
    result = qdb.check_consistency(w, r)
    strong = "✅ Strong" if result["strong_consistency"] else "⚠️  Eventual"
    print(f"  W={w} R={r} (W+R={w+r}) → {strong}  |  {label}")

print("\n  Formula: W + R > N  →  at least one node has latest data")
print("  N=3, W=2, R=2 → 2+2=4 > 3 ✅ (overlap of 1 node guarantees freshness)")

In [ ]:
# ============================================
# 3. PACELC Extension
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════╗
  ║  PACELC: What happens WHEN there's NO partition?               ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║                                                                  ║
  ║  if (Partition)                                                  ║
  ║      choose: Availability vs Consistency     ← CAP              ║
  ║  else                                                            ║
  ║      choose: Latency vs Consistency          ← ELC extension    ║
  ║                                                                  ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  Database     │ P → A/C ?  │ E → L/C ?  │ Full PACELC          ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  DynamoDB     │ PA         │ EL         │ PA/EL (fast always)  ║
  ║  Cassandra    │ PA         │ EL         │ PA/EL (tunable)      ║
  ║  MongoDB      │ PC         │ EC         │ PC/EC (consistent)   ║
  ║  PostgreSQL   │ PC         │ EC         │ PC/EC (ACID)         ║
  ║  CockroachDB  │ PC         │ EC         │ PC/EC (distributed)  ║
  ║  Cosmos DB    │ PA         │ EL/EC      │ PA/EL or PA/EC       ║
  ║               │ (tunable)  │ (tunable)  │ (5 consistency lvls) ║
  ╚══════════════════════════════════════════════════════════════════╝

  Key insight: CAP only applies DURING partitions.
  PACELC also asks: when things are NORMAL, do you prefer
  speed (low Latency) or correctness (Consistency)?
""")

In [ ]:
# ============================================
# 4. Consistency Spectrum
# ============================================

print("""
  Consistency Spectrum (weakest → strongest):

  ┌────────────────────────────────────────────────────────────────┐
  │                                                                │
  │   Eventual  →  Causal  →  Read-your  →  Bounded  →  Strong   │
  │                            -writes      staleness              │
  │                                                                │
  │   DynamoDB    ────────── Cosmos DB ──────────────   PostgreSQL │
  │   Cassandra               (5 levels)                MongoDB   │
  │                                                                │
  └────────────────────────────────────────────────────────────────┘

  1. Eventual:        Data converges "eventually" (ms to seconds)
  2. Causal:          If A causes B, everyone sees A before B
  3. Read-your-writes: You always see your own writes
  4. Bounded staleness: Data at most X seconds old
  5. Strong:           Linearizable, all reads get latest write

  Interview gotcha: "Eventual consistency" doesn't mean "wrong".
  It means the system will converge. For 99.9% of reads,
  data is already consistent (typically < 10ms convergence).
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is the CAP theorem? | In distributed system, during partition you must choose Consistency or Availability |
| 2 | Why can't you have CA in distributed? | Partitions WILL happen in networks. CA only exists for single-node DBs |
| 3 | Explain quorum reads/writes | W+R > N guarantees overlap (strong consistency). N=3, W=2, R=2 is common |
| 4 | What is PACELC? | Extends CAP: during Partition choose A/C, Else choose L/C. Covers normal operations |
| 5 | Eventual consistency problems? | Stale reads, conflict resolution needed (last-write-wins, vector clocks, CRDTs) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • CAP: pick 2 of 3 (C, A, P) during partition          │
## │  • CP: rejects writes during partition (MongoDB, HBase)  │
## │  • AP: serves stale data during partition (Cassandra)    │
## │  • Quorum: W+R > N ensures strong consistency           │
## │  • PACELC adds normal-mode latency vs consistency       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Sharding & Replication** — Scaling databases horizontally